# ✨ บทที่ 6 — Simplify: ทำให้ขอบแปลงสวยงาม

> ต่อจาก: [บทที่ 5 — SAM Production](05_sam_production.ipynb)

---

## ทฤษฎี: 3 วิธีในการ Simplify Polygon

| วิธี | หลักการ | ผลลัพธ์ |
|---|---|---|
| `douglas_peucker` | ลดจำนวนจุดตาม tolerance | เร็ว แต่ยังมีมุมแหลม |
| `buffer_clean` | Dilate (+) แล้ว Erode (-) | เนียน แต่ยังเหลี่ยม |
| `smooth` ✅ | buffer_clean + Chaikin + simplify | โค้งมน ดีที่สุด |

### Chaikin Corner-Cutting Algorithm

```text
จุดเดิม:  P0 ───────────── P1

สูตร:  Q = 0.75×P0 + 0.25×P1  (ใกล้ P0)
       R = 0.25×P0 + 0.75×P1  (ใกล้ P1)

ผล:    P0 ─ Q ─────── R ─ P1  (มุมแหลม → เส้นโค้ง)
```

**Prompt:**
`"A before/after comparison. LEFT: jagged pixelated polygon labeled 'ก่อน Simplify'. RIGHT: same polygon with smooth rounded curves labeled 'หลัง Simplify'. Mint-green #35C778 fill on right. White background, 16:9."`

---

## ขั้นตอนที่ 1 — Setup

In [ ]:
# =====================================================================
# ⚠️ สำหรับรันบน Local Machine (เครื่องตัวเอง)
# =====================================================================
import os
BASE_DIR = r'C:\Users\kittisak\Downloads\ssb tutor\drive_structure\ch06_simplify'


In [ ]:
# =====================================================================
# ☁️ สำหรับรันบน Google Colab (คอมเมนต์ไว้ก่อน)
# =====================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = '/content/drive/MyDrive/ssb_workshop/ch06_simplify'


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.1 — Import และ Paths
# ═══════════════════════════════════════════════════════════

import importlib
import os
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio


In [ ]:
RASTER_DIR = os.path.join(BASE_DIR, 'data')
SAT_PATH   = os.path.join(RASTER_DIR, 'for sam satelite test.tif')
SCRIPT_DIR = os.path.join(BASE_DIR, 'scripts')

# ไฟล์ input: ผลจากบทที่ 5 หรือไฟล์ตัวอย่าง
INPUT_GPKG = os.path.join(BASE_DIR, 'data', 'sam_parcels_final.gpkg')
if not os.path.exists(INPUT_GPKG):
    INPUT_GPKG = os.path.join(RASTER_DIR, 'sam overlay.gpkg')
    print(f'⚠️  ใช้ไฟล์ตัวอย่างแทน: {os.path.basename(INPUT_GPKG)}')

OUTPUT_SIMPLIFIED = os.path.join(BASE_DIR, 'output', 'sam_parcels_simplified.geojson')

gdf_raw = gpd.read_file(INPUT_GPKG)
print(f'✅ โหลดแล้ว: {len(gdf_raw):,} Polygon')

---

## ขั้นตอนที่ 2 — ดูปัญหาก่อน Simplify

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.1 — วิเคราะห์จำนวน Vertices ก่อน Simplify
# ═══════════════════════════════════════════════════════════

def count_vertices(geom):
    if geom is None or geom.is_empty: return 0
    if hasattr(geom, 'exterior'):
        return len(geom.exterior.coords) + sum(len(r.coords) for r in geom.interiors)
    if hasattr(geom, 'geoms'): return sum(count_vertices(g) for g in geom.geoms)
    elif hasattr(geom, 'coords'): return len(geom.coords)
    return 0

vertex_counts = gdf_raw.geometry.apply(count_vertices)
print('📊 ก่อน Simplify:')
print(f'   จำนวน Polygon:             {len(gdf_raw):,}')
print(f'   Vertices เฉลี่ยต่อ Polygon:  {vertex_counts.mean():,.0f} จุด')
print(f'   Vertices รวม:               {vertex_counts.sum():,} จุด')
print(f'\n💡 ยิ่งมีจุดมาก → ไฟล์ขนาดใหญ่ + วาดช้า + เส้นเป็นบันได')

---

## ขั้นตอนที่ 3 — โหลด Script และรัน Simplify

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.1 — โหลด simplify.py จาก Drive
# ═══════════════════════════════════════════════════════════

if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

import simplify
importlib.reload(simplify)
from simplify import clean_geojson_boundaries

print('✅ โหลด simplify.py เรียบร้อย')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.2 — รัน Simplify Pipeline
# ═══════════════════════════════════════════════════════════
# ⏱️ ใช้เวลา 1-5 นาที ขึ้นกับจำนวน Polygon

clean_geojson_boundaries(
    input_path=INPUT_GPKG,
    output_path=OUTPUT_SIMPLIFIED,
    tolerance_meters=2.5,       # ความนุ่มของ smooth (เมตร)
    method='smooth',            # buffer + Chaikin + simplify
    smooth_iterations=2,        # Chaikin 2 รอบ
    min_area_sqm=1600,          # ลบแปลง < 1 ไร่
    resolve_overlaps=True,
    verbose_area_tracking=True, # แสดง % พื้นที่แต่ละขั้น
)

print('✅ Simplify เสร็จสิ้น!')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.3 — เปรียบเทียบก่อน-หลัง
# ═══════════════════════════════════════════════════════════

gdf_simplified = gpd.read_file(OUTPUT_SIMPLIFIED)
v_after = gdf_simplified.geometry.apply(count_vertices)

print('📊 ผลการ Simplify:')
print(f'   ก่อน: {len(gdf_raw):,} Polygon, {vertex_counts.sum():,} vertices')
print(f'   หลัง: {len(gdf_simplified):,} Polygon, {v_after.sum():,} vertices')
print(f'   ลดจำนวนจุดลง: {(1 - v_after.sum()/max(vertex_counts.sum(),1))*100:.1f}%')

with rasterio.open(SAT_PATH) as src:
    bounds = src.bounds
    scale  = min(1.0, 1500 / max(src.width, src.height))
    h, w   = max(1, int(src.height * scale)), max(1, int(src.width * scale))
    rgb    = np.stack([src.read(i, out_shape=(h, w)) for i in range(1, 4)], axis=-1)
    if rgb.max() > 1: rgb = rgb / 255.0
    crs = src.crs

extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(rgb, extent=extent, origin='upper')
gdf_raw.to_crs(crs).boundary.plot(ax=axes[0], color='red', linewidth=0.5, alpha=0.7)
axes[0].set_title(f'ก่อน: {len(gdf_raw):,} Polygon — เส้นบันไดพิกเซล', fontsize=12)
axes[0].axis('off')

axes[1].imshow(rgb, extent=extent, origin='upper')
gdf_simplified.to_crs(crs).boundary.plot(ax=axes[1], color='#35C778', linewidth=0.7)
axes[1].set_title(f'หลัง: {len(gdf_simplified):,} Polygon — เส้นโค้งมนสวยงาม', fontsize=12)
axes[1].axis('off')

plt.suptitle('Simplify: ก่อน vs หลัง', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 🎯 สรุปบทที่ 6

| สิ่งที่เรียนรู้ | รายละเอียด |
|---|---|
| Pixel Staircase | Polygon จาก SAM มีเส้นขอบบันไดจำนวนจุดมาก |
| วิธี `smooth` | buffer → Chaikin → simplify (ดีที่สุด) |
| Chaikin | Q=0.75×P0+0.25×P1, R=0.25×P0+0.75×P1 |
| Pipeline 7 ขั้น | Project→Smooth→Fragment→Overlap→Area→Shape→Export |

> ➡️ **บทที่ 7:** รวม AI + SAM = แผนที่แปลงปาล์มจริง